In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import col, to_timestamp, datediff, when, count, avg, round, desc, rank
from pyspark.sql.window import Window

sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")
sparkConf.setAppName("ShelterBatchPipeline")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")
sparkConf.set("spark.cores.max", "1")

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()

# Configure GCS connector
bucket = "de2026-group10-ass2-data"
spark.conf.set("temporaryGcsBucket", bucket)

conf = spark.sparkContext._jsc.hadoopConfiguration()
conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")

In [3]:
intakes_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"gs://{bucket}/datasets/aac_intakes.csv")

outcomes_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"gs://{bucket}/datasets/aac_outcomes.csv")

# Select, rename overlapping columns, and parse timestamps
intakes_clean = intakes_df.select(
    col("animal_id"),
    col("animal_type"),
    col("breed"),
    col("intake_type"),
    col("intake_condition"),
    to_timestamp(col("datetime")).alias("intake_time")
).dropna(subset=["animal_id", "intake_time"])

outcomes_clean = outcomes_df.select(
    col("animal_id"),
    col("outcome_type"),
    col("outcome_subtype"),
    to_timestamp(col("datetime")).alias("outcome_time")
).dropna(subset=["animal_id", "outcome_time", "outcome_type"])

In [4]:
# Join intakes and outcomes on animal_id, ensuring outcome happened after intake
joined_df = intakes_clean.join(outcomes_clean, on="animal_id", how="inner") \
    .filter(col("outcome_time") >= col("intake_time")) \
    .withColumn("stay_days", datediff(col("outcome_time"), col("intake_time")))

# Aggregation 1: Average length of stay & outcome counts by animal type and outcome
outcome_stats_df = joined_df.groupBy("animal_type", "intake_condition", "outcome_type") \
    .agg(
        count("*").alias("total_animals"),
        round(avg("stay_days"), 2).alias("avg_stay_days")
    )

# Aggregation 2 (Window Function): Rank top breeds by adoption count per animal type
breed_adoptions = joined_df.filter(col("outcome_type") == "Adoption") \
    .groupBy("animal_type", "breed") \
    .agg(count("*").alias("adoption_count"))

windowSpec = Window.partitionBy("animal_type").orderBy(desc("adoption_count"))
top_breeds_df = breed_adoptions.withColumn("breed_rank", rank().over(windowSpec)) \
    .filter(col("breed_rank") <= 10)

In [5]:
project_id = "joekiepoekie-de"

outcome_stats_df.write.format("bigquery") \
    .option("table", f"{project_id}.shelter_analytics.batch_outcome_stats") \
    .mode("overwrite") \
    .save()

top_breeds_df.write.format("bigquery") \
    .option("table", f"{project_id}.shelter_analytics.batch_top_breeds") \
    .mode("overwrite") \
    .save()

spark.stop()